# Course Project Results Summary

这个 notebook 用来汇总 `outputs/course_project_eval_seed42to46_v1` 里的正式结果。

当前默认只看这四个正式数据集：
- `BCIC2A`
- `MDD`
- `SEED`
- `SLEEP`

`CHINESE` 已排除，不纳入最终结果表。

In [1]:
from pathlib import Path
import csv
import math

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 160)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'utils' and NOTEBOOK_DIR.parent.name == 'src':
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

FINAL_OUT_ROOT = PROJECT_ROOT / 'outputs' / 'course_project_eval_seed42to46_v1'
SUMMARY_DIR = FINAL_OUT_ROOT / 'summary_tables'
SHAREABLE_SUMMARY_DIR = PROJECT_ROOT / 'data' / 'course_project' / 'final_results'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
SHAREABLE_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

FORMAL_DATASETS = ['BCIC2A', 'MDD', 'SEED', 'SLEEP']
FINAL_OUT_ROOT

WindowsPath('//10.16.93.90/dataset3/nzh/eeg_FM/reve_eeg/outputs/course_project_eval_seed42to46_v1')

In [2]:
def _to_float(value):
    try:
        if value is None or value == '':
            return math.nan
        return float(value)
    except Exception:
        return math.nan


def load_last_metrics(metrics_path: Path):
    with metrics_path.open('r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return None
    last = rows[-1]
    return {
        'epoch': int(float(last.get('epoch', 0))) if last.get('epoch', '') not in ('', None) else None,
        'best_val_balanced_accuracy': _to_float(last.get('best_val_balanced_accuracy')),
        'val_accuracy': _to_float(last.get('val_accuracy')),
        'val_balanced_accuracy': _to_float(last.get('val_balanced_accuracy')),
        'val_macro_f1': _to_float(last.get('val_macro_f1')),
        'val_weighted_f1': _to_float(last.get('val_weighted_f1')),
        'val_loss': _to_float(last.get('val_loss')),
    }


records = []
for dataset in FORMAL_DATASETS:
    dataset_dir = FINAL_OUT_ROOT / dataset
    if not dataset_dir.exists():
        continue
    for mode_dir in sorted([p for p in dataset_dir.iterdir() if p.is_dir()]):
        mode = mode_dir.name
        for seed_dir in sorted([p for p in mode_dir.iterdir() if p.is_dir()]):
            metrics_path = seed_dir / 'metrics.csv'
            if not metrics_path.exists():
                continue
            row = load_last_metrics(metrics_path)
            if row is None:
                continue
            row.update({
                'dataset': dataset,
                'mode': mode,
                'seed': seed_dir.name,
                'run_dir': str(seed_dir.relative_to(PROJECT_ROOT)).replace('\\', '/'),
                'prediction_csv': (seed_dir / 'test_predictions.csv').exists(),
                'logits_npy': (seed_dir / 'test_logits.npy').exists(),
            })
            records.append(row)

runs_df = pd.DataFrame(records)
runs_df = runs_df.sort_values(['dataset', 'mode', 'seed']).reset_index(drop=True)
runs_df

,epoch,best_val_balanced_accuracy,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,val_loss,dataset,mode,seed,run_dir,prediction_csv,logits_npy
0,17,0.344444,0.313889,0.313889,0.317205,0.317205,3.442860,BCIC2A,ft,seed42,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed42,True,True
1,15,0.372222,0.350000,0.350000,0.348499,0.348499,2.698310,BCIC2A,ft,seed43,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed43,True,True
2,25,0.408333,0.386111,0.386111,0.386871,0.386871,4.154078,BCIC2A,ft,seed44,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed44,True,True
3,11,0.402778,0.363889,0.363889,0.368028,0.368028,2.119830,BCIC2A,ft,seed45,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed45,True,True
4,14,0.361111,0.305556,0.305556,0.306375,0.306375,3.391592,BCIC2A,ft,seed46,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed46,True,True
5,20,0.901563,0.892188,0.892187,0.892175,0.892175,0.580244,MDD,ft,seed42,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed42,True,True
6,17,0.885938,0.884375,0.884375,0.884374,0.884374,0.438164,MDD,ft,seed43,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed43,True,True
7,14,0.900000,0.870313,0.870313,0.870046,0.870046,0.620992,MDD,ft,seed44,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed44,True,True
8,25,0.900000,0.892188,0.892188,0.892181,0.892181,0.643122,MDD,ft,seed45,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed45,True,True
9,13,0.884375,0.876563,0.876562,0.876512,0.876512,0.450325,MDD,ft,seed46,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed46,True,True


In [3]:
summary_cols = [
    'dataset', 'mode', 'seed', 'epoch',
    'best_val_balanced_accuracy', 'val_accuracy', 'val_balanced_accuracy',
    'val_macro_f1', 'val_weighted_f1', 'val_loss',
    'prediction_csv', 'logits_npy', 'run_dir'
]
runs_df[summary_cols]

,dataset,mode,seed,epoch,best_val_balanced_accuracy,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,val_loss,prediction_csv,logits_npy,run_dir
0,BCIC2A,ft,seed42,17,0.344444,0.313889,0.313889,0.317205,0.317205,3.442860,True,True,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed42
1,BCIC2A,ft,seed43,15,0.372222,0.350000,0.350000,0.348499,0.348499,2.698310,True,True,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed43
2,BCIC2A,ft,seed44,25,0.408333,0.386111,0.386111,0.386871,0.386871,4.154078,True,True,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed44
3,BCIC2A,ft,seed45,11,0.402778,0.363889,0.363889,0.368028,0.368028,2.119830,True,True,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed45
4,BCIC2A,ft,seed46,14,0.361111,0.305556,0.305556,0.306375,0.306375,3.391592,True,True,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed46
5,MDD,ft,seed42,20,0.901563,0.892188,0.892187,0.892175,0.892175,0.580244,True,True,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed42
6,MDD,ft,seed43,17,0.885938,0.884375,0.884375,0.884374,0.884374,0.438164,True,True,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed43
7,MDD,ft,seed44,14,0.900000,0.870313,0.870313,0.870046,0.870046,0.620992,True,True,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed44
8,MDD,ft,seed45,25,0.900000,0.892188,0.892188,0.892181,0.892181,0.643122,True,True,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed45
9,MDD,ft,seed46,13,0.884375,0.876563,0.876562,0.876512,0.876512,0.450325,True,True,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed46


In [4]:
agg_df = (
    runs_df.groupby(['dataset', 'mode'], dropna=False)
    .agg(
        n_runs=('seed', 'count'),
        mean_bal_acc=('best_val_balanced_accuracy', 'mean'),
        std_bal_acc=('best_val_balanced_accuracy', 'std'),
        max_bal_acc=('best_val_balanced_accuracy', 'max'),
        mean_macro_f1=('val_macro_f1', 'mean'),
        mean_weighted_f1=('val_weighted_f1', 'mean'),
    )
    .reset_index()
    .sort_values(['dataset', 'max_bal_acc'], ascending=[True, False])
)
agg_df

,dataset,mode,n_runs,mean_bal_acc,std_bal_acc,max_bal_acc,mean_macro_f1,mean_weighted_f1
0,BCIC2A,ft,5,0.377778,0.027287,0.408333,0.345396,0.345396
1,MDD,ft,5,0.894375,0.008458,0.901563,0.883057,0.883057
2,SEED,ft,5,0.411556,0.021222,0.424444,0.372970,0.372970
3,SLEEP,ft,5,0.634298,0.057645,0.735416,0.631479,0.632936


In [5]:
best_seed_df = (
    runs_df.sort_values(
        ['dataset', 'mode', 'best_val_balanced_accuracy', 'val_macro_f1', 'val_weighted_f1'],
        ascending=[True, True, False, False, False],
    )
    .groupby(['dataset', 'mode'], as_index=False)
    .first()
    .sort_values(['dataset', 'best_val_balanced_accuracy'], ascending=[True, False])
)
best_seed_df[['dataset', 'mode', 'seed', 'best_val_balanced_accuracy', 'val_macro_f1', 'val_weighted_f1', 'run_dir']]

,dataset,mode,seed,best_val_balanced_accuracy,val_macro_f1,val_weighted_f1,run_dir
0,BCIC2A,ft,seed44,0.408333,0.386871,0.386871,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed44
1,MDD,ft,seed42,0.901563,0.892175,0.892175,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed42
2,SEED,ft,seed44,0.424444,0.400472,0.400472,outputs/course_project_eval_seed42to46_v1/SEED/ft/seed44
3,SLEEP,ft,seed42,0.735416,0.728820,0.729300,outputs/course_project_eval_seed42to46_v1/SLEEP/ft/seed42


In [6]:
dataset_best_df = (
    runs_df.sort_values(
        ['dataset', 'best_val_balanced_accuracy', 'val_macro_f1', 'val_weighted_f1'],
        ascending=[True, False, False, False],
    )
    .groupby('dataset', as_index=False)
    .first()
)
dataset_best_df[['dataset', 'mode', 'seed', 'best_val_balanced_accuracy', 'val_macro_f1', 'val_weighted_f1', 'run_dir']]

,dataset,mode,seed,best_val_balanced_accuracy,val_macro_f1,val_weighted_f1,run_dir
0,BCIC2A,ft,seed44,0.408333,0.386871,0.386871,outputs/course_project_eval_seed42to46_v1/BCIC2A/ft/seed44
1,MDD,ft,seed42,0.901563,0.892175,0.892175,outputs/course_project_eval_seed42to46_v1/MDD/ft/seed42
2,SEED,ft,seed44,0.424444,0.400472,0.400472,outputs/course_project_eval_seed42to46_v1/SEED/ft/seed44
3,SLEEP,ft,seed42,0.735416,0.728820,0.729300,outputs/course_project_eval_seed42to46_v1/SLEEP/ft/seed42


In [7]:
runs_csv = SUMMARY_DIR / 'formal_runs_all.csv'
agg_csv = SUMMARY_DIR / 'formal_aggregate_by_dataset_mode.csv'
best_seed_csv = SUMMARY_DIR / 'formal_best_seed_by_dataset_mode.csv'
dataset_best_csv = SUMMARY_DIR / 'formal_best_overall_by_dataset.csv'

share_runs_csv = SHAREABLE_SUMMARY_DIR / 'formal_runs_all.csv'
share_agg_csv = SHAREABLE_SUMMARY_DIR / 'formal_aggregate_by_dataset_mode.csv'
share_best_seed_csv = SHAREABLE_SUMMARY_DIR / 'formal_best_seed_by_dataset_mode.csv'
share_dataset_best_csv = SHAREABLE_SUMMARY_DIR / 'formal_best_overall_by_dataset.csv'

runs_df.to_csv(runs_csv, index=False, encoding='utf-8-sig')
agg_df.to_csv(agg_csv, index=False, encoding='utf-8-sig')
best_seed_df.to_csv(best_seed_csv, index=False, encoding='utf-8-sig')
dataset_best_df.to_csv(dataset_best_csv, index=False, encoding='utf-8-sig')

runs_df.to_csv(share_runs_csv, index=False, encoding='utf-8-sig')
agg_df.to_csv(share_agg_csv, index=False, encoding='utf-8-sig')
best_seed_df.to_csv(share_best_seed_csv, index=False, encoding='utf-8-sig')
dataset_best_df.to_csv(share_dataset_best_csv, index=False, encoding='utf-8-sig')

print('Saved:')
print(runs_csv)
print(agg_csv)
print(best_seed_csv)
print(dataset_best_csv)
print(share_runs_csv)
print(share_agg_csv)
print(share_best_seed_csv)
print(share_dataset_best_csv)

Saved:
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\outputs\course_project_eval_seed42to46_v1\summary_tables\formal_runs_all.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\outputs\course_project_eval_seed42to46_v1\summary_tables\formal_aggregate_by_dataset_mode.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\outputs\course_project_eval_seed42to46_v1\summary_tables\formal_best_seed_by_dataset_mode.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\outputs\course_project_eval_seed42to46_v1\summary_tables\formal_best_overall_by_dataset.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\data\course_project\final_results\formal_runs_all.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\data\course_project\final_results\formal_aggregate_by_dataset_mode.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\data\course_project\final_results\formal_best_seed_by_dataset_mode.csv
\\10.16.93.90\dataset3\nzh\eeg_FM\reve_eeg\data\course_project\final_results\formal_best_overall_by_dataset.csv
